# Companion notebook: Cryptography Decoded: Algebra 1: Elements of Number Theory

© 2001–2026 Gerald Teschl (http://www.mat.univie.ac.at/~gerald/) and Susanne Teschl (http://staff.technikum-wien.at/~teschl/)

This [Python](https://python.org) notebook is intended as a supplement to our book "[Cryptography Decoded](https://www.mat.univie.ac.at/~gerald/ftp/book-cry/)." Additional explanations can also be found there.
You can easily install the code from [PyPI](https://pypi.org/project/kryptools/).

## Prime numbers

To get a list of primes up to a given bound, we can use the sieve of Eratosthenes. The idea is to start with the smallest prime, $2$, and cross out all its multiples. The next number that hasn't been crossed out is the next prime: all the numbers crossed out so far are composite, and if this next number had a smaller prime factor, it would already have been crossed out. We then cross out the multiples of this new prime and repeat. We only need to process primes up to the square root of the bound, since any composite number within the bound has a prime factor no larger than that. For a little extra efficiency, the code below starts at $3$ and only considers odd numbers:

In [1]:
def sieve_eratosthenes(max: int) -> list:
    """Returns a list of all primes up to (including) max."""
    is_prime = [True] * (max + 1)  # to begin with, all numbers are potentially prime

    p = 3  # we ignore the prime 2 for now
    while p * p < max:
        if is_prime[p]:  # sieve out non-primes; note that numbers p*q with q<p were already sieved out previously
            for j in range(p * p, max + 1, 2 * p):
                is_prime[j] = False
        p += 2

    primes = [2]  # now we only need to add the odd ones
    for p in range(3, max + 1, 2):
        if is_prime[p]:
            primes.append(p)

    return primes

sieve_eratosthenes(50)

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]

To find the prime factorization of $n$, we only need to check prime factors $p\leq \sqrt{n}$: if we write $n$ as a product of two factors, they can't both be $> \sqrt{n}$. We can either precompute the primes we need or do everything in one go:

In [2]:
from math import isqrt


def factor_eratosthenes(n: int, multiple: bool = False) -> list:
    """Factors a given number n using the sieve of Eratosthenes. With multiple=True/False gives a list/dict of primes."""
    factors = []  # we first handle multiples of 2
    while n % 2 == 0:
        factors.append(2)
        n = n // 2
    max = isqrt(n - 1) + 1 # =floor(sqrt(n))
    is_prime = [True] * (max + 1)

    p = 3  # now we continue sieving out the remaining primes
    while p <= max:  # we only search for prime factors with p*p <= n
        if is_prime[p]:
            while n % p == 0:
                factors.append(p)
                n = n // p
            for j in range(p * p, max + 1, p):  # sieve out multiples of p
                is_prime[j] = False
        p += 2

    if n != 1:  # one primefactor could be larger than max
        factors.append(n)

    if multiple == True:  # list of prime factors with multiplicities
        return factors

    factor_dict = {}  # dict of prime factors (values) with multiplicities (keys)
    for p in factors:
        if p in factor_dict:
            factor_dict[p] += 1
        else:
            factor_dict[p] = 1
    return factor_dict


factor_eratosthenes(12)

{2: 2, 3: 1}

The result is a dictionary (`dict`), with the prime factors as keys and their multiplicities as values. With the option `multiple=True`, we get a list of prime factors:

In [3]:
factor_eratosthenes(12, True)

[2, 2, 3]

The sieve of Eratosthenes is slow and only suitable for small numbers. We'll see more efficient methods later. Here's a preview of the result:

In [4]:
from kryptools import factorint

factorint(1400)

{2: 3, 5: 2, 7: 1}

Let's check:

In [5]:
from math import prod

prod(p**k for p, k in _.items())

1400

To check whether a number is prime, we can use the following command:

In [6]:
from kryptools import is_prime

is_prime(1400)

False

In Python, we can use `divmod` to get both the quotient and the remainder:

In [7]:
divmod(17, 5)

(3, 2)

Alternatively, we can get the integer quotient and the remainder separately:

In [8]:
17 // 5, 17 % 5

(3, 2)

Keep in mind: the division operator `/` returns a floating-point number, even when the division has no remainder:

In [9]:
17 / 5

3.4

We can import functions for the greatest common divisor and the least common multiple:

In [10]:
from math import gcd, lcm

gcd(8, 12), lcm(630, 180)

(4, 1260)

## Modular arithmetic

We already know how to calculate remainders. We can use this to build an addition table for $\mathbb{Z}_n$:

In [11]:
n = 5
[[(a + b) % n for a in range(n)] for b in range(n)]

[[0, 1, 2, 3, 4],
 [1, 2, 3, 4, 0],
 [2, 3, 4, 0, 1],
 [3, 4, 0, 1, 2],
 [4, 0, 1, 2, 3]]

We can find the additive inverse of $2$ in $\mathbb{Z}_n$ with

In [12]:
-2 % n

3

and the multiplicative inverse with

In [13]:
pow(2, -1, n)

3

In general, `pow(a,k,n)` computes the power $a^k \pmod{n}$.

## A class for $\mathbb{Z}_n$

We can also define a class for $\mathbb{Z}_n$. More precisely, we'll define two classes. The first, `Zmod`, specifies the ring by setting the modulus $n$.

In [14]:
class Zmod:
    "Ring of intergers modulo n."

    def __init__(self, n: int, short: bool = True):
        self.n = n
        self.short = short

    def __call__(self, x: int):
        return ZmodPoint(x, self)

    def __eq__(self, other):
        if isinstance(other, self.__class__):
            return self.n == other.n
        return False

    def __contains__(self, other: "ZmodPoint") -> bool:
        return isinstance(other, ZmodPoint) and self.n == other.ring.n

The second class, `ZmodPoint`, represents the elements of the ring. Each element stores its representative $x$ and the ring it belongs to. We also overload the relevant operators so that the usual arithmetic operations work as expected.

In [15]:
class ZmodPoint:
    "Represents a point in the ring Zmod."

    def __init__(self, x: int, ring: "Zmod"):
        self.x = int(x) % ring.n
        self.ring = ring

    def __repr__(self):
        if self.ring.short:
            return str(self.x)
        return f"{self.x} (mod {self.ring.n})"

    def __eq__(self, other):
        if not isinstance(other, self.__class__) or self.ring != other.ring:
            return False
        return self.x == other.x

    def __int__(self):
        return self.x

    def __add__(self, other: "ZmodPoint") -> "ZmodPoint":
        if isinstance(other, self.__class__) and self.ring == other.ring:
            return self.__class__(self.x + other.x, self.ring)
        if isinstance(other, int):
            return self.__class__(self.x + other, self.ring)
        return NotImplemented

    def __radd__(self, scalar: int) -> "ZmodPoint":
        return self.__class__(scalar + self.x, self.ring)

    def __neg__(self) -> "ZmodPoint":
        return self.__class__(-self.x, self.ring)

    def __sub__(self, other: "ZmodPoint") -> "ZmodPoint":
        if isinstance(other, self.__class__) and self.ring == other.ring:
            return self.__class__(self.x - other.x, self.ring)
        if isinstance(other, int):
            return self.__class__(self.x - other, self.ring)
        return NotImplemented

    def __rsub__(self, scalar: int) -> "ZmodPoint":
        return self.__class__(scalar - self.x, self.ring)

    def __mul__(self, other: "ZmodPoint") -> "ZmodPoint":
        if isinstance(other, self.__class__) and self.ring == other.ring:
            return self.__class__(self.x * other.x, self.ring)
        if isinstance(other, int):
            return self.__class__(self.x * other, self.ring)
        return NotImplemented

    def __rmul__(self, scalar: int) -> "ZmodPoint":
        return self.__class__(scalar * self.x, self.ring)

    def __truediv__(self, other: "ZmodPoint") -> "ZmodPoint":
        if not isinstance(other, self.__class__) or self.ring != other.ring:
            return NotImplemented
        return self.__class__(self.x * pow(other.x, -1, self.ring.n), self.ring)

    def __rtruediv__(self, scalar: int) -> "ZmodPoint":
        return self.__class__(scalar * pow(self.x, -1, self.ring.n), self.ring)

    def __pow__(self, scalar: int) -> "ZmodPoint":
        return self.__class__(pow(self.x, scalar, self.ring.n), self.ring)

Now we can do some calculations:

In [16]:
Z_5 = Zmod(5, short = False)

a = Z_5(3)
b = Z_5(4)

a + b

2 (mod 5)

In [17]:
a * a**-1

1 (mod 5)

The class is also available in our package, with a few extra features. For example, we can solve the linear equation $a x = b \pmod{m}$:

In [18]:
from kryptools import Zmod

Z_30= Zmod(30)

a, b = Z_30([18, 24])
a.solve(b, all_solutions = True)

[3, 8, 13, 18, 23, 28]

We can calculate Euler's phi function using the formula based on the prime factorization:

In [19]:
def euler_phi(n: int) -> int:
    """Euler's phi function of `n`."""
    k = factorint(n)
    return prod([(p - 1) * p ** (k[p] - 1) for p in k])


euler_phi(20)

8

We can also list the elements of $\Z_m$ and $Z_m^*$:

In [20]:
list(Zmod(4))

[0, 1, 2, 3]

In [21]:
list(Zmod(4).star())

[1, 3]

## The Euclidean algorithm

The (extended) Euclidean algorithm is easy to implement in Python, for example using recursion:

In [22]:
def gcd(a: int, b: int) -> int:
    """Compute the greatest common divisor of two integers a and b."""
    if b < 0:
        b *= -1
    if a == 0:
        return b
    if a < 0:
        a *= -1
    return gcd(b % a, a)

def egcd(a: int, b: int) -> (int, int, int):
    """Perform the extended Euclidean agorithm. Returns gcd, x, y such that a x + b y = gcd."""
    if a == 0:
        return b, 0, 1
    else:
        gcd, x, y = egcd(b % a, a)
        if gcd < 0:
            return -gcd, (b // a) * x - y, -x
        return gcd, y - (b // a) * x, x


egcd(-39, -17)

(1, -7, 16)

Here's an alternative version without recursion:

In [23]:
def egcd(a: int, b: int) -> (int, int, int):
    """Perform the extended Euclidean agorithm. Returns gcd, x, y such that a x + b y = gcd."""
    if a == 0:
        if b < 0:
            return -b, 0, -1
        return b, 0, 1
    if a < 0:    
        r0, r1 = -a, b
        x0, x1, y0, y1 = -1, 0, 0, 1
    else:
        r0, r1 = a, b
        x0, x1, y0, y1 = 1, 0, 0, 1
    if b < 0:
        r1 *= -1
        y1 = -1
    while r1 != 0:
        q, r = divmod(r0, r1)
        r0, r1 = r1, r
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
    return r0, x0, y0


egcd(39, 17)

(1, 7, -16)

Let's check:

In [24]:
39 * _[1] + 17 * _[2] == _[0]

True

We can also use this to calculate the modular inverse:

In [25]:
def inverse(a: int, n: int) -> int:
    """Compute the multiplicative inverse in Z_n. Returns x such that a*x % n = 1."""
    gcd, x, y = egcd(n, a % n)
    if gcd != 1:
        raise ValueError(f"{a} is not invertible mod {n}.")
    return y % n

inverse(17, 39)

23

## The Chinese remainder theorem

We can solve a system of congruences using the Chinese remainder theorem:

In [26]:
def crt(a: list[int], m: list[int]) -> int:
    """Solve given linear congruences x % m[j] == a[j] using the Chinese Remainder Theorem."""
    l = len(a)
    if len(m) != l:
        raise ValueError("The lists of numbers and modules must have equal length.")
    M = prod(m)
    Mi = [M // m[i] for i in range(l)]
    MNi = [Mi[i] * pow(Mi[i], -1, m[i]) % M for i in range(l)]
    return sum([a[i] * MNi[i] % M for i in range(l)]) % M


a = [2, 3, 2]
m = [3, 5, 7]
crt(a, m)

23

Let's check:

In [27]:
assert [_ % mi for mi in m] == a

The version in the package also handles the case where the moduli are not coprime:

In [28]:
from kryptools import crt

m = [12, 30]
a = [3, 21]
crt(a, m, coprime = False)

51

Let's check:

In [29]:
assert [_ % mi for mi in m] == a

## The Asmuth–Bloom scheme

First, we generate suitable parameters. We choose primes for the moduli $m_j$ to ensure that they are coprime.

In [30]:
from math import prod
from kryptools import next_prime, crt

n, r = 5, 3
m = 2**32 # max size of the secret

M = []
mm = 2 * m
for _ in range(n):
    mm = next_prime(mm + 1)
    M.append(mm)

M1 = prod(M[:r])
assert m * prod(M[1-r:]) < M1
M

[8589934609, 8589934621, 8589934627, 8589934631, 8589934651]

Now we can split a secret $s$ into $n$ shares:

In [31]:
from random import randint, sample, seed
seed(0)

s = randint(0, m - 1)
t = randint(1, (M1 - s) // m)
x = s + t * m
assert x < M1

A = [ x % mm for mm in M]
A

[953188841, 2267150034, 7376225206, 341660855, 3090211041]

Let's check that we can reconstruct the secret:

In [32]:
smpl = sample(range(n), r)

assert crt([ A[i] for i in smpl ], [ M[i] for i in smpl ]) % m == s

## Vector spaces over finite fields

We can use the `Matrix` class to work with vectors and matrices.

In [33]:
from kryptools import Matrix, Zmod

A = Matrix([[3, 2], [5, -1], [0, 7]])
B = Matrix([[1, 4], [0, -3]])

A * B

[ 3,   6 ]
[ 5,  23 ]
[ 0, -21 ]

The transpose:

In [34]:
A.transpose()

[ 3,  5, 0 ]
[ 2, -1, 7 ]

We can also calculate the kernel,

In [35]:
Z_5 = Zmod(5)

A = Matrix([[1, 2, 3], [0, 0, 1], [0, 0, 0]], ring = Z_5)
A.kernel()

[ 3 ]
[ 1 ]
[ 0 ]

the image,

In [36]:
A.img()

[ 1, 0 ]
[ 0, 1 ]
[ 0, 0 ]

the rank,

In [37]:
A.rank()

2

and the determinant:

In [38]:
A.det()

0

We can find the inverse with

In [39]:
B.inv()

[ 1,  4/3 ]
[ 0, -1/3 ]

Let's check:

In [40]:
B * _

[ 4,  16 ]
[ 0, -12 ]